# Load environment


In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Part 1: Configuration
Adjust these before running

In [ ]:
CONFIG = {

    # ── Data ──────────────────────────────────────────────────────────────────
    "state_cols":       [],         # list of preprocessed feature column names
                                    # e.g. ["anchor_age_norm", "ecg_acuity", ...]
    "action_col":       "",         # column name for the action taken
                                    # e.g. "icu_care"
    "reward_col":       "",         # column name for the observed reward
                                    # e.g. "reward"
    "next_state_cols":  [],         # column names for the next state
                                    # e.g. ["next_anchor_age_norm", ...]
    "done_col":         "",         # column indicating episode termination
                                    # (1 = terminal, 0 = continuing)

    # ── Action space ──────────────────────────────────────────────────────────
    "n_actions":        None,       # number of discrete actions
                                    # e.g. 2 for binary (transfer / don't transfer)

    # ── Network architecture ───────────────────────────────────────────────────
    "hidden_layers":    [256, 256], # hidden layer sizes for all networks
                                    # BCQ and distributional nets share this
    "activation":       "relu",     # "relu", "tanh", or "elu"

    # ── BCQ-specific ──────────────────────────────────────────────────────────
    "latent_dim":       32,         # VAE latent space dimensionality
                                    # typically state_dim // 2 is a good start
    "n_candidates":     10,         # number of actions sampled from VAE
                                    # per forward pass at inference
                                    # higher = better coverage, slower inference
    "vae_beta":         0.5,        # KL divergence weight in VAE loss
                                    # higher = more regularized latent space
    "vae_kl_clip":      0.5,        # clamp on VAE latent std to prevent
                                    # posterior collapse during early training
    "perturbation_phi": 0.05,       # max perturbation magnitude applied to
                                    # VAE-sampled actions by the perturbation net
                                    # smaller = stays closer to behavior policy

    # ── Distributional Q-specific (QR-DQN) ────────────────────────────────────
    "n_quantiles":      51,         # number of quantile atoms
                                    # more = finer return distribution
                                    # 51 (C51) and 200 (QR-DQN) are common choices
    "huber_kappa":      1.0,        # Huber loss threshold in quantile regression
                                    # controls transition between L1 and L2 loss

    # ── Training ──────────────────────────────────────────────────────────────
    "learning_rate":    1e-4,       # shared learning rate for all optimizers
    "vae_learning_rate":1e-3,       # VAE can tolerate a higher learning rate
                                    # than the Q-network
    "batch_size":       64,
    "n_epochs":         100,
    "gamma":            0.99,       # discount factor
    "tau":              0.005,      # soft target network update rate
                                    # BCQ typically uses smaller tau than CQL
    "grad_clip":        1.0,        # gradient clipping max norm (None to disable)

    # ── Regularization ────────────────────────────────────────────────────────
    "dropout":          0.0,        # dropout between hidden layers (0 = off)
    "weight_decay":     1e-4,       # L2 regularization on Q-network optimizer

    # ── Risk preference (distributional) ──────────────────────────────────────
    "risk_measure":     "mean",     # how to select actions from the distribution
                                    # "mean"  — maximize expected return (neutral)
                                    # "cvar"  — minimize conditional value at risk
                                    #           (conservative, good for clinical)
                                    # "worst" — maximize worst-case quantile
    "cvar_alpha":       0.25,       # CVaR tail fraction (only used if
                                    # risk_measure = "cvar")
                                    # 0.25 = focus on bottom 25% of outcomes

    # ── Reproducibility ───────────────────────────────────────────────────────
    "seed":             42,

    # ── Output ────────────────────────────────────────────────────────────────
    "model_save_path":  "bcq_distributional.pt",
    "log_every_n":      10,
}

# Part 2: Data Loading
Load the dataset here

In [ ]:
# ── Replace with your actual dataframes ───────────────────────────────────────
# Examples:
#   df_train = pd.read_parquet("train.parquet")
#   df_val   = pd.read_parquet("val.parquet")
#   df_test  = pd.read_parquet("test.parquet")

df_train = None   # ← replace
df_val   = None   # ← replace
df_test  = None   # ← replace

# Part 3: Tensor Conversion

In [ ]:
torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

def df_to_tensors(df):
    s  = torch.tensor(df[CONFIG["state_cols"]].values.astype(np.float32))
    a  = torch.tensor(df[CONFIG["action_col"]].values.astype(np.int64))
    r  = torch.tensor(df[CONFIG["reward_col"]].values.astype(np.float32))
    ns = torch.tensor(df[CONFIG["next_state_cols"]].values.astype(np.float32))
    d  = torch.tensor(df[CONFIG["done_col"]].values.astype(np.float32))
    return s, a, r, ns, d

s_train, a_train, r_train, ns_train, d_train = df_to_tensors(df_train)
s_val,   a_val,   r_val,   ns_val,   d_val   = df_to_tensors(df_val)
s_test,  a_test,  r_test,  ns_test,  d_test  = df_to_tensors(df_test)

train_loader = DataLoader(
    TensorDataset(s_train, a_train, r_train, ns_train, d_train),
    batch_size=CONFIG["batch_size"],
    shuffle=True,
)

# Part 4: Model Architecture
Three networks are trained jointly:
1) VAE - learns the behavior policy (what actions were taken)
2) Distributional Q-network + target - estimates return distributions
3) Perturbation network - fine-tunes VAE-sampled actions

In [ ]:
def build_activation(name):
    return {"relu": nn.ReLU(), "tanh": nn.Tanh(), "elu": nn.ELU()}[name]

def build_mlp(in_dim, out_dim, hidden_layers, activation, dropout):
    """Shared MLP builder used by all three networks."""
    layers = []
    for h in hidden_layers:
        layers.append(nn.Linear(in_dim, h))
        layers.append(build_activation(activation))
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        in_dim = h
    layers.append(nn.Linear(in_dim, out_dim))
    return nn.Sequential(*layers)

# ── 4a. VAE — behavior policy generative model ────────────────────────────────
# The VAE learns which actions were taken in each state by the behavior policy
# (clinicians). At inference, it generates candidate actions for a given state
# that are constrained to the data distribution — preventing OOD action selection.

class VAE(nn.Module):
    def __init__(self, state_dim, n_actions, latent_dim, hidden_layers,
                 activation, dropout):
        super().__init__()
        self.latent_dim = latent_dim
        self.n_actions  = n_actions

        # Encoder: (state + one-hot action) → (mu, log_var) in latent space
        # One-hot encoding keeps the action representation continuous for the VAE
        enc_in = state_dim + n_actions
        self.encoder_body = build_mlp(
            enc_in, hidden_layers[-1], hidden_layers[:-1], activation, dropout
        )
        self.mu      = nn.Linear(hidden_layers[-1], latent_dim)
        self.log_var = nn.Linear(hidden_layers[-1], latent_dim)

        # Decoder: (state + latent_z) → reconstructed action logits
        self.decoder = build_mlp(
            state_dim + latent_dim, n_actions, hidden_layers, activation, dropout
        )

    def encode(self, state, action_onehot):
        h       = self.encoder_body(torch.cat([state, action_onehot], dim=1))
        mu      = self.mu(h)
        log_var = self.log_var(h)
        return mu, log_var

    def reparameterize(self, mu, log_var):
        # Clamp std to prevent posterior collapse in early training
        std = torch.exp(0.5 * log_var).clamp(max=CONFIG["vae_kl_clip"])
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, state, z):
        return self.decoder(torch.cat([state, z], dim=1))

    def forward(self, state, action_onehot):
        mu, log_var = self.encode(state, action_onehot)
        z           = self.reparameterize(mu, log_var)
        recon_logits = self.decode(state, z)
        return recon_logits, mu, log_var

    def sample_actions(self, state, n_candidates):
        """
        Sample n_candidates actions from the prior for a given state.
        Used at inference to generate the candidate action set.
        Returns action indices of shape (batch, n_candidates).
        """
        batch_size  = state.shape[0]
        state_rep   = state.repeat_interleave(n_candidates, dim=0)
        z           = torch.randn(batch_size * n_candidates,
                                  self.latent_dim).clamp(-0.5, 0.5)
        logits      = self.decode(state_rep, z)
        actions     = logits.argmax(dim=1).view(batch_size, n_candidates)
        return actions


# ── 4b. Distributional Q-Network (QR-DQN) ─────────────────────────────────────
# Instead of outputting a single expected Q-value per action, this network
# outputs N_QUANTILES values per action, representing the full return distribution.
# This allows risk-sensitive action selection and provides uncertainty estimates
# that can be displayed on the clinical dashboard.

class DistributionalQNetwork(nn.Module):
    def __init__(self, state_dim, n_actions, n_quantiles, hidden_layers,
                 activation, dropout):
        super().__init__()
        self.n_actions   = n_actions
        self.n_quantiles = n_quantiles

        self.net = build_mlp(
            state_dim,
            n_actions * n_quantiles,   # flattened output — reshaped in forward()
            hidden_layers,
            activation,
            dropout,
        )

    def forward(self, state):
        batch = state.shape[0]
        out   = self.net(state)
        # Reshape to (batch, n_actions, n_quantiles)
        # Each action has its own distribution over n_quantiles return atoms
        return out.view(batch, self.n_actions, self.n_quantiles)

    def q_values(self, state, risk_measure="mean", cvar_alpha=0.25):
        """
        Collapse the return distribution to a scalar Q-value per action
        using the configured risk measure.

        risk_measure options:
          "mean"  — standard expected return (risk-neutral)
          "cvar"  — Conditional Value at Risk: mean of the bottom alpha quantiles
                    (conservative — appropriate for clinical settings)
          "worst" — minimum quantile (maximally pessimistic)
        """
        quantiles = self.forward(state)   # (batch, n_actions, n_quantiles)

        if risk_measure == "mean":
            return quantiles.mean(dim=2)

        elif risk_measure == "cvar":
            # Take the mean over the bottom cvar_alpha fraction of quantiles
            n_tail = max(1, int(cvar_alpha * self.n_quantiles))
            tail, _ = quantiles.topk(n_tail, dim=2, largest=False)
            return tail.mean(dim=2)

        elif risk_measure == "worst":
            return quantiles.min(dim=2).values

        else:
            raise ValueError(f"Unknown risk_measure: {risk_measure}. "
                             f"Choose 'mean', 'cvar', or 'worst'.")


# ── 4c. Perturbation network ───────────────────────────────────────────────────
# After the VAE samples candidate actions, the perturbation network makes small
# adjustments to each candidate to improve Q-value without leaving the
# behavior policy's support. phi controls the maximum perturbation magnitude.

class PerturbationNetwork(nn.Module):
    def __init__(self, state_dim, n_actions, hidden_layers, activation,
                 dropout, phi):
        super().__init__()
        self.phi      = phi
        self.n_actions = n_actions
        # Input: (state + action one-hot) → perturbation logits over actions
        self.net = build_mlp(
            state_dim + n_actions, n_actions, hidden_layers, activation, dropout
        )

    def forward(self, state, action_onehot):
        raw         = self.net(torch.cat([state, action_onehot], dim=1))
        # Scale perturbation to [-phi, phi] — stays close to behavior policy
        perturbation = torch.tanh(raw) * self.phi
        perturbed    = action_onehot + perturbation
        return torch.softmax(perturbed, dim=1)   # keep as valid distribution


# ── Instantiate all networks ───────────────────────────────────────────────────

state_dim = len(CONFIG["state_cols"])

vae = VAE(
    state_dim        = state_dim,
    n_actions        = CONFIG["n_actions"],
    latent_dim       = CONFIG["latent_dim"],
    hidden_layers    = CONFIG["hidden_layers"],
    activation       = CONFIG["activation"],
    dropout          = CONFIG["dropout"],
)

q_network = DistributionalQNetwork(
    state_dim     = state_dim,
    n_actions     = CONFIG["n_actions"],
    n_quantiles   = CONFIG["n_quantiles"],
    hidden_layers = CONFIG["hidden_layers"],
    activation    = CONFIG["activation"],
    dropout       = CONFIG["dropout"],
)

target_q_network = DistributionalQNetwork(
    state_dim     = state_dim,
    n_actions     = CONFIG["n_actions"],
    n_quantiles   = CONFIG["n_quantiles"],
    hidden_layers = CONFIG["hidden_layers"],
    activation    = CONFIG["activation"],
    dropout       = CONFIG["dropout"],
)
target_q_network.load_state_dict(q_network.state_dict())
target_q_network.eval()

perturbation_net = PerturbationNetwork(
    state_dim     = state_dim,
    n_actions     = CONFIG["n_actions"],
    hidden_layers = CONFIG["hidden_layers"],
    activation    = CONFIG["activation"],
    dropout       = CONFIG["dropout"],
    phi           = CONFIG["perturbation_phi"],
)

# ── Optimizers ────────────────────────────────────────────────────────────────
vae_optimizer = optim.Adam(
    vae.parameters(),
    lr=CONFIG["vae_learning_rate"],
)
q_optimizer = optim.Adam(
    list(q_network.parameters()) + list(perturbation_net.parameters()),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)

# Part 5: Loss Function
Bellman MSE loss and CQL penalty

In [ ]:
# ── 5a. VAE loss ──────────────────────────────────────────────────────────────
# Reconstruction loss: how well the decoder recovers the original action
# KL loss: how close the latent distribution is to the standard normal prior
# vae_beta controls the tradeoff — higher = more regularized latent space

def vae_loss(recon_logits, true_actions, mu, log_var):
    recon_loss = nn.CrossEntropyLoss()(recon_logits, true_actions)
    kl_loss    = -0.5 * (1 + log_var - mu.pow(2) - log_var.exp()).mean()
    return recon_loss + CONFIG["vae_beta"] * kl_loss, recon_loss, kl_loss


# ── 5b. Quantile regression loss (QR-DQN Bellman) ────────────────────────────
# Replaces MSE Bellman loss with quantile Huber loss.
# Each quantile atom is updated independently toward the distributional target.
# This gives the network a full picture of return uncertainty, not just the mean.

def quantile_huber_loss(pred_quantiles, target_quantiles):
    """
    pred_quantiles   : (batch, n_quantiles) — predicted quantile values
    target_quantiles : (batch, n_quantiles) — Bellman target quantile values
    """
    n_quantiles = CONFIG["n_quantiles"]
    kappa       = CONFIG["huber_kappa"]

    # Quantile levels tau_i = (2i - 1) / (2N) for i = 1..N
    tau = ((torch.arange(n_quantiles).float() + 0.5) / n_quantiles)

    # Pairwise differences: (batch, pred_quantiles, target_quantiles)
    diff = target_quantiles.unsqueeze(1) - pred_quantiles.unsqueeze(2)

    # Huber loss element-wise
    huber = torch.where(
        diff.abs() <= kappa,
        0.5 * diff.pow(2),
        kappa * (diff.abs() - 0.5 * kappa),
    )

    # Asymmetric quantile weighting
    weight = (tau - (diff < 0).float()).abs()
    loss   = (weight * huber).mean(dim=2).mean(dim=1).mean()
    return loss


def compute_q_loss(states, actions, rewards, next_states, dones):
    """
    Full BCQ distributional Bellman loss:
      1. Generate candidate next-actions from VAE
      2. Perturb candidates with perturbation network
      3. Select best candidate using target Q-network (BCQ constraint)
      4. Compute distributional Bellman target
      5. Compute quantile Huber loss on current Q-network
    """
    batch_size  = states.shape[0]
    n_quantiles = CONFIG["n_quantiles"]

    with torch.no_grad():
        # ── Step 1: sample candidate actions from VAE for next states ─────────
        cand_actions = vae.sample_actions(
            next_states, CONFIG["n_candidates"]
        )   # (batch, n_candidates)

        # ── Step 2: perturb candidates ────────────────────────────────────────
        # Flatten to (batch * n_candidates) for batch processing
        ns_rep     = next_states.repeat_interleave(CONFIG["n_candidates"], dim=0)
        cand_flat  = cand_actions.view(-1)
        cand_onehot = nn.functional.one_hot(
            cand_flat, CONFIG["n_actions"]
        ).float()
        perturbed_onehot = perturbation_net(ns_rep, cand_onehot)

        # ── Step 3: select best candidate using target Q (BCQ constraint) ─────
        # BCQ only selects among VAE-plausible actions — not all possible actions
        target_quantiles_all = target_q_network(ns_rep)  # (batch*cands, n_a, n_q)
        # Score each candidate using the configured risk measure
        target_q_vals = target_q_network.q_values(
            ns_rep,
            risk_measure=CONFIG["risk_measure"],
            cvar_alpha=CONFIG["cvar_alpha"],
        )   # (batch*cands, n_actions)

        # Pick Q-value of the perturbed candidate action for each sample
        cand_q = (target_q_vals * perturbed_onehot).sum(dim=1)
        cand_q = cand_q.view(batch_size, CONFIG["n_candidates"])

        # Best candidate index per batch item
        best_idx    = cand_q.argmax(dim=1)   # (batch,)
        best_actions = cand_actions.gather(
            1, best_idx.unsqueeze(1)
        ).squeeze(1)    # (batch,)

        # ── Step 4: build distributional Bellman target ───────────────────────
        # Get quantile distribution for best next actions
        target_q_dist = target_q_network(next_states)   # (batch, n_actions, n_q)
        best_expanded = best_actions.view(-1, 1, 1).expand(-1, 1, n_quantiles)
        next_quantiles = target_q_dist.gather(1, best_expanded).squeeze(1)
        # (batch, n_quantiles)

        bellman_targets = (
            rewards.unsqueeze(1) +
            CONFIG["gamma"] * next_quantiles * (1 - dones).unsqueeze(1)
        )   # (batch, n_quantiles)

    # ── Step 5: quantile Huber loss on current Q-network ─────────────────────
    current_q_dist = q_network(states)   # (batch, n_actions, n_quantiles)
    actions_exp    = actions.view(-1, 1, 1).expand(-1, 1, n_quantiles)
    pred_quantiles = current_q_dist.gather(1, actions_exp).squeeze(1)
    # (batch, n_quantiles)

    loss = quantile_huber_loss(pred_quantiles, bellman_targets)
    return loss

# Part 6: Training Loop
- VAE and Q-network are trained joinely each epoch.
- The VAE is trained first so the Q-network always has an up-to-date behavior model to contrain against.

In [ ]:
def soft_update(q_net, target_net, tau):
    for q_param, t_param in zip(q_net.parameters(), target_net.parameters()):
        t_param.data.copy_(tau * q_param.data + (1 - tau) * t_param.data)

train_losses_vae = []
train_losses_q   = []
val_losses_q     = []

for epoch in range(1, CONFIG["n_epochs"] + 1):

    vae.train()
    q_network.train()
    perturbation_net.train()

    epoch_vae_loss = 0.0
    epoch_q_loss   = 0.0

    for s, a, r, ns, d in train_loader:

        # ── Train VAE ─────────────────────────────────────────────────────────
        a_onehot = nn.functional.one_hot(a, CONFIG["n_actions"]).float()
        recon_logits, mu, log_var = vae(s, a_onehot)
        v_loss, recon, kl = vae_loss(recon_logits, a, mu, log_var)

        vae_optimizer.zero_grad()
        v_loss.backward()
        if CONFIG["grad_clip"] is not None:
            nn.utils.clip_grad_norm_(vae.parameters(), CONFIG["grad_clip"])
        vae_optimizer.step()
        epoch_vae_loss += v_loss.item()

        # ── Train distributional Q + perturbation network ─────────────────────
        q_loss = compute_q_loss(s, a, r, ns, d)

        q_optimizer.zero_grad()
        q_loss.backward()
        if CONFIG["grad_clip"] is not None:
            nn.utils.clip_grad_norm_(
                list(q_network.parameters()) +
                list(perturbation_net.parameters()),
                CONFIG["grad_clip"],
            )
        q_optimizer.step()
        soft_update(q_network, target_q_network, CONFIG["tau"])
        epoch_q_loss += q_loss.item()

    avg_vae = epoch_vae_loss / len(train_loader)
    avg_q   = epoch_q_loss   / len(train_loader)
    train_losses_vae.append(avg_vae)
    train_losses_q.append(avg_q)

    # ── Validation ────────────────────────────────────────────────────────────
    vae.eval()
    q_network.eval()
    perturbation_net.eval()

    with torch.no_grad():
        val_q_loss = compute_q_loss(s_val, a_val, r_val, ns_val, d_val)
    val_losses_q.append(val_q_loss.item())

    if epoch % CONFIG["log_every_n"] == 0:
        print(
            f"Epoch {epoch:>4} | "
            f"VAE: {avg_vae:.4f} | "
            f"Q train: {avg_q:.4f} | "
            f"Q val: {val_q_loss.item():.4f}"
        )

# Part 7: Evaluation

In [ ]:
vae.eval()
q_network.eval()
perturbation_net.eval()

with torch.no_grad():

    # ── Test loss ─────────────────────────────────────────────────────────────
    test_q_loss = compute_q_loss(s_test, a_test, r_test, ns_test, d_test)

    # ── Return distribution on test set ───────────────────────────────────────
    test_quantiles = q_network(s_test)       # (batch, n_actions, n_quantiles)
    test_q_mean    = test_quantiles.mean(dim=2)   # expected Q per action
    test_q_std     = test_quantiles.std(dim=2)    # uncertainty per action

    # ── Greedy actions under configured risk measure ───────────────────────────
    greedy_actions = q_network.q_values(
        s_test,
        risk_measure=CONFIG["risk_measure"],
        cvar_alpha=CONFIG["cvar_alpha"],
    ).argmax(dim=1)

print(f"\n── Test set results ─────────────────────────────")
print(f"  Q loss (quantile Huber): {test_q_loss.item():.4f}")

# ── Policy-clinician agreement ────────────────────────────────────────────────
agreement = (greedy_actions == a_test).float().mean().item()
print(f"  Policy-clinician agreement: {agreement:.1%}")

# ── Q-value distribution summary ──────────────────────────────────────────────
print(f"\n── Mean Q-values per action (test set) ──────────")
print(pd.DataFrame(test_q_mean.numpy()).describe().round(4))

print(f"\n── Q-value uncertainty (std) per action (test set) ──")
print(pd.DataFrame(test_q_std.numpy()).describe().round(4))

# ── Action distributions ──────────────────────────────────────────────────────
print(f"\n── Greedy policy action distribution ───────────")
for idx, count in zip(*np.unique(greedy_actions.numpy(), return_counts=True)):
    print(f"  Action {idx}: {count} ({count / len(greedy_actions):.1%})")

print(f"\n── Clinician action distribution (test set) ────")
for idx, count in zip(*np.unique(a_test.numpy(), return_counts=True)):
    print(f"  Action {idx}: {count} ({count / len(a_test):.1%})")

# Part 8: Save and Inference

In [ ]:
torch.save({
    "vae_state_dict":              vae.state_dict(),
    "q_network_state_dict":        q_network.state_dict(),
    "target_q_network_state_dict": target_q_network.state_dict(),
    "perturbation_state_dict":     perturbation_net.state_dict(),
    "vae_optimizer_state_dict":    vae_optimizer.state_dict(),
    "q_optimizer_state_dict":      q_optimizer.state_dict(),
    "config":                      CONFIG,
    "train_losses_vae":            train_losses_vae,
    "train_losses_q":              train_losses_q,
    "val_losses_q":                val_losses_q,
    "test_q_loss":                 test_q_loss.item(),
}, CONFIG["model_save_path"])

print(f"\nModel saved to {CONFIG['model_save_path']}")

# ── Inference on a new patient ────────────────────────────────────────────────
# Features must be preprocessed to match training scale before calling this.
# Returns the recommended action, Q-values, and uncertainty per action —
# the uncertainty dict is directly usable for dashboard confidence intervals.

def recommend_action(patient_features: dict) -> dict:
    """
    patient_features : dict with keys matching CONFIG["state_cols"]
                       values must already be preprocessed to training scale

    returns : {
        "recommended_action" : int   — action index with highest risk-adjusted Q
        "q_values"           : dict  — expected Q per action
        "uncertainty"        : dict  — return std per action (for dashboard CI)
        "quantiles"          : dict  — full quantile distribution per action
    }
    """
    raw   = np.array(
                [[patient_features[col] for col in CONFIG["state_cols"]]],
                dtype=np.float32
            )
    state = torch.tensor(raw)

    vae.eval()
    q_network.eval()
    perturbation_net.eval()

    with torch.no_grad():
        quantiles = q_network(state)          # (1, n_actions, n_quantiles)
        q_mean    = quantiles.mean(dim=2)     # (1, n_actions)
        q_std     = quantiles.std(dim=2)      # (1, n_actions)

        risk_q    = q_network.q_values(
            state,
            risk_measure=CONFIG["risk_measure"],
            cvar_alpha=CONFIG["cvar_alpha"],
        )
        best_action = int(risk_q.argmax(dim=1).item())

    return {
        "recommended_action": best_action,
        "q_values":   {i: round(float(v), 4)
                       for i, v in enumerate(q_mean.squeeze(0).tolist())},
        "uncertainty":{i: round(float(v), 4)
                       for i, v in enumerate(q_std.squeeze(0).tolist())},
        "quantiles":  {i: [round(float(q), 4) for q in qs]
                       for i, qs in enumerate(
                           quantiles.squeeze(0).tolist()
                       )},
    }

# Example usage (fill in actual preprocessed feature values):
# result = recommend_action({
#     "anchor_age_norm":   0.62,
#     "ecg_acuity":        2,
#     "rad_acuity_level":  1,
#     ...
# })
# print(result["recommended_action"])
# print(result["uncertainty"])   # feed directly into dashboard confidence display